# **Energy Consumption Forecasting Pipeline — Complete Test Suite**

Built directly from `bronze_ingestion.ipynb`, `silver_layer.ipynb`, and `gold_layer.ipynb`.

**Business rules from scripts:**
- `adjustment_amount` **can be negative** — billing credits
- `load_variation` / `frequency_variation` **can be negative** — grid measurements
- `efficiency_ratio` **clamped to [0.5, 1.0]** in Silver device_metrics
- Silver text cols are **UPPER-cased** and trimmed
- `condition_type` has `%` stripped → values are CLOUDY / RAINY / SUNNY
- `event_timestamp` = parsed `timestamp` column (format `dd-MM-yyyy HH:mm`) in silver_energy_usage
- Gold fact uses surrogate keys: `household_sk`, `device_sk`, `location_sk`, `tariff_sk`
- Gold derived cols: `energy_cost`, `power_factor`, `device_efficiency`, `grid_stress_ratio`

## **Cell 1 — Imports & Catalog Config**

In [0]:
from pyspark.sql.functions import col, sum as spark_sum

CAT = 'energy_catalog'
BR  = 'raw'
SV  = 'processed'
GO  = 'analytics'

# Exact table names from the actual scripts
src_tables = {
    'src_energy_usage'   : f'{CAT}.{BR}.src_energy_usage',
    'src_device_metrics' : f'{CAT}.{BR}.src_device_metrics',
    'src_grid_load'      : f'{CAT}.{BR}.src_grid_load',
    'src_tariff_metrics' : f'{CAT}.{BR}.src_tariff_metrics',
    'src_weather'        : f'{CAT}.{BR}.src_weather',
}

bronze_tables = {
    'bronze_energy_usage'   : f'{CAT}.{BR}.bronze_energy_usage',
    'bronze_device_metrics' : f'{CAT}.{BR}.bronze_device_metrics',
    'bronze_grid_load'      : f'{CAT}.{BR}.bronze_grid_load',
    'bronze_tariff_metrics' : f'{CAT}.{BR}.bronze_tariff_metrics',
    'bronze_weather'        : f'{CAT}.{BR}.bronze_weather',
}

silver_tables = {
    'silver_energy_usage'   : f'{CAT}.{SV}.silver_energy_usage',
    'silver_device_metrics' : f'{CAT}.{SV}.silver_device_metrics',
    'silver_grid_load'      : f'{CAT}.{SV}.silver_grid_load',
    'silver_tariff_metrics' : f'{CAT}.{SV}.silver_tariff_metrics',
    'silver_weather'        : f'{CAT}.{SV}.silver_weather',
}

gold_tables = {
    'dim_household'          : f'{CAT}.{GO}.dim_household',
    'dim_device'             : f'{CAT}.{GO}.dim_device',
    'dim_location'           : f'{CAT}.{GO}.dim_location',
    'dim_tariff'             : f'{CAT}.{GO}.dim_tariff',
    'fact_energy_consumption': f'{CAT}.{GO}.fact_energy_consumption',
}

metrics_tables = {
    'metrics_regional_energy_summary'    : f'{CAT}.{GO}.metrics_regional_energy_summary',
    'metrics_household_consumption_tiers': f'{CAT}.{GO}.metrics_household_consumption_tiers',
    'metrics_device_efficiency_ranking'  : f'{CAT}.{GO}.metrics_device_efficiency_ranking',
    'metrics_tariff_plan_comparison'     : f'{CAT}.{GO}.metrics_tariff_plan_comparison',
    'metrics_grid_health_overview'       : f'{CAT}.{GO}.metrics_grid_health_overview',
    'metrics_weather_energy_correlation' : f'{CAT}.{GO}.metrics_weather_energy_correlation',
}

def load(t): return spark.table(t)

def assert_not_empty(t):
    cnt = load(t).count()
    assert cnt > 0, f'FAILED: {t} is empty'
    print(f'PASSED: {t} — {cnt:,} records')

def assert_no_nulls(t, cols):
    df = load(t)
    for c in cols:
        n = df.filter(col(c).isNull()).count()
        assert n == 0, f'FAILED: {t}.{c} has {n} nulls'
        print(f'PASSED: {t}.{c} — no nulls')

def assert_no_duplicates(t, key_cols):
    dup = load(t).groupBy(key_cols).count().filter(col('count') > 1).count()
    assert dup == 0, f'FAILED: {t} has {dup} duplicates on {key_cols}'
    print(f'PASSED: {t} — no duplicates on {key_cols}')

def assert_non_negative(t, cols):
    df = load(t)
    for c in cols:
        neg = df.filter(col(c) < 0).count()
        assert neg == 0, f'FAILED: {t}.{c} has {neg} negative values'
        print(f'PASSED: {t}.{c} — no negative values')

print('Config loaded successfully')

---
# **Part 1 — Data Quality Checks**

### **All tables not empty**

In [0]:
for tbl in {**bronze_tables, **silver_tables, **gold_tables, **metrics_tables}.values():
    assert_not_empty(tbl)

### **Bronze — audit columns present, no null household_id**

In [0]:
AUDIT_COLS = ['_batch_date', '_ingestion_ts', '_source_file', '_stream_name']

for name, tbl_path in bronze_tables.items():
    df = load(tbl_path)
    for ac in AUDIT_COLS:
        assert ac in df.columns, f'FAILED: {name} missing audit column {ac}'
    print(f'PASSED: {name} — all 4 audit columns present')
    null_hh = df.filter(col('household_id').isNull()).count()
    assert null_hh == 0, f'FAILED: {name}.household_id has {null_hh} nulls'
    print(f'PASSED: {name}.household_id — no nulls')

### **Silver — nulls, duplicates, non-negatives**

> **Note:** `adjustment_amount`, `load_variation`, `frequency_variation` are intentionally excluded from non-negative checks — they hold valid negative values per script logic.

In [0]:
# silver_energy_usage
# event_timestamp = parsed timestamp (dd-MM-yyyy HH:mm)
assert_no_nulls(silver_tables['silver_energy_usage'],
    ['household_id', 'region_name', 'energy_usage_kwh',
     'active_power_kw', 'event_timestamp'])
# silver_energy_usage uses full-row dropDuplicates() — no surrogate key
# silver_energy_usage uses full-row dropDuplicates() — no surrogate key
# (household_id, event_timestamp) is NOT unique — multiple readings per household per time
eu_total    = load(silver_tables["silver_energy_usage"]).count()
eu_distinct = load(silver_tables["silver_energy_usage"]).distinct().count()
assert eu_total == eu_distinct, \
    f"FAILED: silver_energy_usage has {eu_total - eu_distinct} fully duplicate rows"
print(f"PASSED: silver_energy_usage — no full-row duplicates ({eu_total:,} rows, all distinct)")
assert_non_negative(silver_tables['silver_energy_usage'],
    ['energy_usage_kwh', 'active_power_kw', 'daily_consumption_kwh',
     'voltage_reading', 'current_reading'])

# silver_device_metrics — efficiency_ratio clamped to [0.5, 1.0]
assert_no_nulls(silver_tables['silver_device_metrics'],
    ['household_id', 'device_category', 'device_brand', 'device_model'])
# silver_device_metrics dedup key matches the script:
# (household_id, device_category, device_brand, device_model, runtime_hours_rounded)
# runtime_hours_rounded is dropped after dedup, so we recreate it here to verify

from pyspark.sql.functions import round as spark_round

dm_df = load(silver_tables["silver_device_metrics"]) \
    .withColumn("runtime_hours_rounded", spark_round("runtime_hours", 0))

dup_cnt = (
    dm_df.groupBy(
        "household_id", "device_category", "device_brand",
        "device_model", "runtime_hours_rounded"
    )
    .count()
    .filter(col("count") > 1)
    .count()
)
assert dup_cnt == 0, \
    f"FAILED: silver_device_metrics has {dup_cnt} duplicates on script dedup key"
print(f"PASSED: silver_device_metrics — no duplicates on (household_id, device_category, device_brand, device_model, runtime_hours_rounded)")
assert_non_negative(silver_tables['silver_device_metrics'],
    ['energy_draw_kwh', 'runtime_hours', 'device_power_kw'])

# silver_grid_load
# load_variation & frequency_variation: NOT tested for non-negative (signed grid values)
assert_no_nulls(silver_tables['silver_grid_load'],
    ['household_id', 'grid_region', 'grid_load_kw', 'grid_capacity_kw'])
assert_no_duplicates(silver_tables['silver_grid_load'],
    ['household_id', 'grid_region', 'substation_name',
     'feeder_line', 'distribution_zone', 'grid_operator'])
assert_non_negative(silver_tables['silver_grid_load'],
    ['grid_load_kw', 'grid_capacity_kw', 'reserve_margin', 'demand_forecast_kw'])

# silver_tariff_metrics
# adjustment_amount: NOT tested for non-negative (billing credits)
assert_no_nulls(silver_tables['silver_tariff_metrics'],
    ['household_id', 'tariff_plan_type', 'billing_cycle', 'monthly_bill'])
assert_no_duplicates(silver_tables['silver_tariff_metrics'],
    ['household_id', 'tariff_region', 'tariff_city',
     'tariff_plan_type', 'utility_provider'])
assert_non_negative(silver_tables['silver_tariff_metrics'],
    ['monthly_bill', 'unit_rate', 'billing_units',
     'peak_rate', 'offpeak_rate', 'fixed_charge'])

# silver_weather — condition_type cleaned (% removed → CLOUDY/RAINY/SUNNY)
assert_no_nulls(silver_tables['silver_weather'],
    ['household_id', 'weather_region', 'temperature_celsius',
     'humidity_percent', 'condition_type'])
# silver_weather uses full-row dropDuplicates() — no unique key
# (household_id, timestamp) is NOT unique — multiple weather readings per household per day
wt_total    = load(silver_tables["silver_weather"]).count()
wt_distinct = load(silver_tables["silver_weather"]).distinct().count()
assert wt_total == wt_distinct, \
    f"FAILED: silver_weather has {wt_total - wt_distinct} fully duplicate rows"
print(f"PASSED: silver_weather — no full-row duplicates ({wt_total:,} rows, all distinct)")

### **Gold — Dimension tables: unique surrogate keys**

In [0]:
# dim_household — SK: household_sk, NK: household_id
assert_no_nulls(gold_tables['dim_household'],
    ['household_sk', 'household_id', 'meter_type',
     'customer_category', 'region_name', 'city_name'])
assert_no_duplicates(gold_tables['dim_household'], ['household_sk'])
assert_no_duplicates(gold_tables['dim_household'], ['household_id'])

# dim_device — SK: device_sk
assert_no_nulls(gold_tables['dim_device'],
    ['device_sk', 'device_category', 'device_brand', 'device_model'])
assert_no_duplicates(gold_tables['dim_device'], ['device_sk'])

# dim_location — SK: location_sk, NK: region_name
assert_no_nulls(gold_tables['dim_location'],
    ['location_sk', 'region_name', 'city_name'])
assert_no_duplicates(gold_tables['dim_location'], ['location_sk'])
assert_no_duplicates(gold_tables['dim_location'], ['region_name'])

# dim_tariff — SK: tariff_sk, NK: tariff_plan_type
assert_no_nulls(gold_tables['dim_tariff'],
    ['tariff_sk', 'tariff_plan_type'])
assert_no_duplicates(gold_tables['dim_tariff'], ['tariff_sk'])
assert_no_duplicates(gold_tables['dim_tariff'], ['tariff_plan_type'])

### **Gold — Fact table: nulls, non-negatives, referential integrity via surrogate keys**

In [0]:
assert_no_nulls(gold_tables['fact_energy_consumption'],
    ['fact_sk', 'household_sk', 'location_sk',
     'energy_usage_kwh', 'active_power_kw',
     'peak_demand_kw', 'offpeak_demand_kw'])
assert_non_negative(gold_tables['fact_energy_consumption'],
    ['energy_usage_kwh', 'active_power_kw',
     'peak_demand_kw', 'offpeak_demand_kw',
     'monthly_bill', 'billing_units', 'energy_cost'])

fact    = load(gold_tables['fact_energy_consumption'])
dim_hh  = load(gold_tables['dim_household'])
dim_loc = load(gold_tables['dim_location'])
dim_dev = load(gold_tables['dim_device'])
dim_tar = load(gold_tables['dim_tariff'])

for dim_df, sk, label in [
    (dim_hh,  'household_sk', 'dim_household'),
    (dim_loc, 'location_sk',  'dim_location'),
    (dim_dev, 'device_sk',    'dim_device'),
    (dim_tar, 'tariff_sk',    'dim_tariff'),
]:
    bad = fact.join(dim_df, sk, 'left_anti').count()
    assert bad == 0, f'FAILED: fact has {bad} invalid {sk} refs to {label}'
    print(f'PASSED: fact_energy_consumption → {label} ({sk}) integrity')

### **Silver → Gold reconciliation**

In [0]:
silver_eu = load(silver_tables['silver_energy_usage'])
s_cnt = silver_eu.count()
g_cnt = fact.count()
assert s_cnt == g_cnt, f'FAILED: silver_energy_usage ({s_cnt:,}) != fact ({g_cnt:,})'
print(f'PASSED: row count matches — silver={s_cnt:,}, fact={g_cnt:,}')

s_sum = silver_eu.agg(spark_sum('energy_usage_kwh').alias('t')).collect()[0]['t']
g_sum = fact.agg(spark_sum('energy_usage_kwh').alias('t')).collect()[0]['t']
assert round(s_sum, 2) == round(g_sum, 2), \
    f'FAILED: energy_usage_kwh sum — silver={s_sum}, gold={g_sum}'
print(f'PASSED: energy_usage_kwh sum — silver={s_sum:.2f}, gold={g_sum:.2f}')

### **Metrics tables — business rules**

In [0]:
# metrics_regional_energy_summary — grain: (region_name, city_name, climate_zone)
assert_no_nulls(metrics_tables['metrics_regional_energy_summary'],
    ['region_name', 'avg_energy_usage_kwh', 'total_energy_usage_kwh'])
assert_non_negative(metrics_tables['metrics_regional_energy_summary'],
    ['avg_energy_usage_kwh', 'total_energy_usage_kwh', 'unique_households'])

# metrics_household_consumption_tiers — grain: household_sk
# consumption_tier: Low / Medium / High / Very High
assert_no_nulls(metrics_tables['metrics_household_consumption_tiers'],
    ['household_sk', 'household_id', 'consumption_tier'])
assert_no_duplicates(metrics_tables['metrics_household_consumption_tiers'],
    ['household_sk'])
bad_tiers = load(metrics_tables['metrics_household_consumption_tiers']) \
    .filter(~col('consumption_tier').isin('Low', 'Medium', 'High', 'Very High')).count()
assert bad_tiers == 0, f'FAILED: {bad_tiers} rows have invalid consumption_tier'
print('PASSED: metrics_household_consumption_tiers — all consumption_tier values valid (Low/Medium/High/Very High)')

# metrics_device_efficiency_ranking — grain: (device_category, device_brand)
# efficiency_grade: A / B / C / D
assert_no_nulls(metrics_tables['metrics_device_efficiency_ranking'],
    ['device_category', 'device_brand', 'avg_efficiency_ratio'])
assert_non_negative(metrics_tables['metrics_device_efficiency_ranking'],
    ['avg_efficiency_ratio', 'avg_runtime_hours', 'avg_power_kw'])
bad_grades = load(metrics_tables['metrics_device_efficiency_ranking']) \
    .filter(~col('efficiency_grade').isin('A', 'B', 'C', 'D')).count()
assert bad_grades == 0, f'FAILED: {bad_grades} rows have invalid efficiency_grade'
print('PASSED: metrics_device_efficiency_ranking — all efficiency_grade values valid (A/B/C/D)')

# metrics_tariff_plan_comparison — grain: tariff_plan_type
assert_no_nulls(metrics_tables['metrics_tariff_plan_comparison'],
    ['tariff_plan_type', 'avg_energy_cost', 'avg_monthly_bill'])
assert_non_negative(metrics_tables['metrics_tariff_plan_comparison'],
    ['enrolled_households', 'avg_usage_kwh', 'total_usage_kwh'])

# metrics_grid_health_overview and metrics_weather_energy_correlation
assert_not_empty(metrics_tables['metrics_grid_health_overview'])
assert_not_empty(metrics_tables['metrics_weather_energy_correlation'])

print('ALL DATA QUALITY CHECKS PASSED SUCCESSFULLY')

---
# **Part 2 — Unit Tests: Key Transformation Logic**

Every test maps directly to a step in the Silver or Gold scripts.

In [0]:
print('=' * 65)
print('UNIT TESTS — KEY TRANSFORMATION LOGIC')
print('=' * 65)

eu = load(silver_tables['silver_energy_usage'])
dm = load(silver_tables['silver_device_metrics'])
gl = load(silver_tables['silver_grid_load'])
tm = load(silver_tables['silver_tariff_metrics'])
wt = load(silver_tables['silver_weather'])

# ── 1. event_timestamp derived column exists and has no nulls ─────────────
# Script: to_timestamp(col('timestamp'), 'dd-MM-yyyy HH:mm') → event_timestamp
assert 'event_timestamp' in eu.columns, \
    'FAILED: silver_energy_usage missing event_timestamp column'
null_ts = eu.filter(col('event_timestamp').isNull()).count()
assert null_ts == 0, f'FAILED: event_timestamp has {null_ts} nulls — parsing failed'
print('PASSED: silver_energy_usage.event_timestamp — exists, no nulls (dd-MM-yyyy HH:mm parsed)')

# ── 2. Audit columns dropped in every Silver table ────────────────────────
# Script: df.drop('_batch_date', '_source_file', '_stream_name', '_ingestion_ts')
for sv_name, sv_path in silver_tables.items():
    df_s = load(sv_path)
    for ac in ['_batch_date', '_ingestion_ts', '_source_file', '_stream_name']:
        assert ac not in df_s.columns, \
            f'FAILED: {sv_name} still has audit col {ac}'
print('PASSED: All Silver tables — 4 audit columns correctly dropped')

# ── 3. Dedup — silver_energy_usage (full-row dropDuplicates, ~5 rows removed) ──
# Script: df.dropDuplicates()  ← no key, full-row only
# (household_id, event_timestamp) is NOT a unique key — multiple readings per slot
eu_total    = eu.count()
eu_distinct = eu.distinct().count()
assert eu_total == eu_distinct, \
    f'FAILED: silver_energy_usage has {eu_total - eu_distinct} full-row duplicates after dedup'
src_cnt = load(src_tables['src_energy_usage']).count()
assert eu_total <= src_cnt, \
    f'FAILED: silver_energy_usage ({eu_total:,}) > src ({src_cnt:,})'
print(f'PASSED: silver_energy_usage — full-row dedup correct, {src_cnt - eu_total} rows removed (expected ~5)')

# ── 4. Dedup — silver_device_metrics (keyed dedup with runtime_hours_rounded) ─
# Script: df.withColumn('runtime_hours_rounded', spark_round('runtime_hours', 0))
#         df.dropDuplicates(['household_id', 'device_category', 'device_brand',
#                            'device_model', 'runtime_hours_rounded'])
#         .drop('runtime_hours_rounded')
from pyspark.sql.functions import round as spark_round

dm_keyed = dm.withColumn('runtime_hours_rounded', spark_round('runtime_hours', 0))
dm_dup = (
    dm_keyed.groupBy(
        'household_id', 'device_category', 'device_brand',
        'device_model', 'runtime_hours_rounded'
    )
    .count()
    .filter(col('count') > 1)
    .count()
)
assert dm_dup == 0, \
    f'FAILED: silver_device_metrics has {dm_dup} duplicates on script dedup key'
print('PASSED: silver_device_metrics — no duplicates on (household_id, device_category, device_brand, device_model, runtime_hours_rounded)')

# ── 5. Dedup — silver_grid_load (infrastructure identity key) ────────────
# Script: df.dropDuplicates(['household_id', 'grid_region', 'substation_name',
#                            'feeder_line', 'distribution_zone', 'grid_operator'])
gl_dup = (
    gl.groupBy(
        'household_id', 'grid_region', 'substation_name',
        'feeder_line', 'distribution_zone', 'grid_operator'
    )
    .count()
    .filter(col('count') > 1)
    .count()
)
assert gl_dup == 0, \
    f'FAILED: silver_grid_load has {gl_dup} duplicates on infrastructure identity key'
print('PASSED: silver_grid_load — no duplicates on infrastructure identity key')

# ── 6. Dedup — silver_tariff_metrics (business key, ~951 rows removed) ───
# Script: df.dropDuplicates(['household_id', 'tariff_region', 'tariff_city',
#                            'tariff_plan_type', 'utility_provider'])
tm_dup = (
    tm.groupBy(
        'household_id', 'tariff_region', 'tariff_city',
        'tariff_plan_type', 'utility_provider'
    )
    .count()
    .filter(col('count') > 1)
    .count()
)
assert tm_dup == 0, \
    f'FAILED: silver_tariff_metrics has {tm_dup} duplicates on business key'
print('PASSED: silver_tariff_metrics — no duplicates on (household_id, tariff_region, tariff_city, tariff_plan_type, utility_provider)')

# ── 7. Dedup — silver_weather (full-row dropDuplicates, 0 rows removed) ──
# Script: df.dropDuplicates()  ← no key, full-row only
# (household_id, timestamp) is NOT a unique key — multiple weather readings per slot
wt_total    = wt.count()
wt_distinct = wt.distinct().count()
assert wt_total == wt_distinct, \
    f'FAILED: silver_weather has {wt_total - wt_distinct} full-row duplicates after dedup'
print(f'PASSED: silver_weather — full-row dedup correct, 0 rows removed (expected ~0)')

# ── 8. Null drop — energy_usage: 10 numeric cols must have 0 nulls ────────
# Script: dropna(subset=[...10 numeric cols...]) — ~1000 rows dropped
for c in ['voltage_reading', 'current_reading', 'active_power_kw',
          'reactive_power_kvar', 'energy_usage_kwh', 'frequency_hz',
          'load_factor', 'peak_demand_kw', 'offpeak_demand_kw',
          'daily_consumption_kwh']:
    n = eu.filter(col(c).isNull()).count()
    assert n == 0, f'FAILED: silver_energy_usage.{c} still has {n} nulls'
print('PASSED: silver_energy_usage — all 10 numeric cols: 0 nulls after dropna')

# ── 9. efficiency_ratio clamped to [0.5, 1.0] ─────────────────────────────
# Script: when(col < 0.5, 0.5).when(col > 1.0, 1.0).otherwise(col)
assert dm.filter(col('efficiency_ratio') < 0.5).count() == 0, \
    'FAILED: efficiency_ratio values below 0.5 after clamping'
assert dm.filter(col('efficiency_ratio') > 1.0).count() == 0, \
    'FAILED: efficiency_ratio values above 1.0 after clamping'
print('PASSED: silver_device_metrics.efficiency_ratio — clamped to [0.5, 1.0]')

# ── 10. load_variation & frequency_variation CAN be negative ─────────────
# Script comment: 'legitimate grid measurements — NOT errors'
neg_lv = gl.filter(col('load_variation') < 0).count()
neg_fv = gl.filter(col('frequency_variation') < 0).count()
assert neg_lv > 0, 'FAILED: no negative load_variation — expected signed values (-20 to +20)'
assert neg_fv > 0, 'FAILED: no negative frequency_variation — expected signed values (-0.5 to +0.5)'
print(f'PASSED: silver_grid_load — {neg_lv:,} neg load_variation, {neg_fv:,} neg freq_variation preserved')

# ── 11. adjustment_amount CAN be negative — billing credits ───────────────
# Script comment: 'Negative values = billing credits — NOT errors'
neg_adj = tm.filter(col('adjustment_amount') < 0).count()
assert neg_adj > 0, 'FAILED: no negative adjustment_amount — billing credits should exist'
print(f'PASSED: silver_tariff_metrics — {neg_adj:,} billing credit rows preserved')

# ── 12. condition_type has no % characters ────────────────────────────────
# Script: regexp_replace(col('condition_type'), '%', '')
pct = wt.filter(col('condition_type').contains('%')).count()
assert pct == 0, f'FAILED: {pct} rows still have % in condition_type'
print('PASSED: silver_weather.condition_type — % suffix stripped')

# ── 13. condition_type values are UPPER: CLOUDY, RAINY, SUNNY ─────────────
invalid = wt.filter(~col('condition_type').isin('CLOUDY', 'RAINY', 'SUNNY')).count()
assert invalid == 0, f'FAILED: {invalid} rows have unexpected condition_type values'
print('PASSED: silver_weather.condition_type — only CLOUDY/RAINY/SUNNY present')

# ── 14. Gold derived columns exist ────────────────────────────────────────
for derived_col in ['energy_cost', 'power_factor', 'device_efficiency', 'grid_stress_ratio']:
    assert derived_col in fact.columns, \
        f'FAILED: fact_energy_consumption missing {derived_col}'
    print(f'PASSED: fact_energy_consumption.{derived_col} — derived column exists')

null_ec = fact.filter(col('energy_cost').isNull()).count()
neg_ec  = fact.filter(col('energy_cost') < 0).count()
assert null_ec == 0 and neg_ec == 0, \
    f'FAILED: energy_cost has {null_ec} nulls and {neg_ec} negatives'
print('PASSED: fact.energy_cost — no nulls, no negatives')

# ── 15. Surrogate keys present and not null ───────────────────────────────
for sk in ['fact_sk', 'household_sk', 'device_sk', 'location_sk', 'tariff_sk']:
    assert sk in fact.columns, f'FAILED: fact missing surrogate key {sk}'
    n = fact.filter(col(sk).isNull()).count()
    assert n == 0, f'FAILED: fact.{sk} has {n} null values'
    print(f'PASSED: fact_energy_consumption.{sk} — exists, no nulls')

print()
print('ALL UNIT TESTS FOR KEY TRANSFORMATION LOGIC PASSED')

---
# **Part 3 — Schema Validation**

Expected schemas built directly from `StructType` definitions in `bronze_ingestion.ipynb` and `.select()` / `.withColumn()` calls in `gold_layer.ipynb`.

In [0]:
print('=' * 65)
print('SCHEMA VALIDATION')
print('=' * 65)

NUM = {'double', 'float'}
INT = {'int', 'integer', 'bigint', 'long'}
STR = {'string'}
STR_OR_INT = {'string', 'int', 'integer', 'bigint', 'long'}
TS  = {'timestamp', 'string'}

expected_schemas = {
    # ── Bronze ───────────────────────────────────────────────────────────────
    bronze_tables['bronze_energy_usage']: {
        'household_id': STR_OR_INT, 'region_name': STR, 'city_name': STR,
        'meter_type': STR, 'customer_category': STR, 'grid_zone': STR,
        'voltage_reading': NUM, 'current_reading': NUM, 'active_power_kw': NUM,
        'reactive_power_kvar': NUM, 'energy_usage_kwh': NUM, 'frequency_hz': NUM,
        'load_factor': NUM, 'peak_demand_kw': NUM, 'offpeak_demand_kw': NUM,
        'daily_consumption_kwh': NUM, 'timestamp': STR,
        '_batch_date': STR, '_ingestion_ts': TS, '_source_file': STR, '_stream_name': STR,
    },
    bronze_tables['bronze_device_metrics']: {
        'household_id': STR_OR_INT, 'device_category': STR, 'device_brand': STR,
        'device_model': STR, 'maintenance_status': STR, 'installation_region': STR,
        'runtime_hours': NUM, 'device_power_kw': NUM, 'motor_speed_rpm': NUM,
        'efficiency_ratio': NUM, 'energy_draw_kwh': NUM, 'heat_output': NUM,
        'cooling_load': NUM, 'device_voltage': NUM, 'device_current': NUM,
        'device_temperature': NUM,
        '_batch_date': STR, '_ingestion_ts': TS, '_source_file': STR, '_stream_name': STR,
    },
    bronze_tables['bronze_grid_load']: {
        'household_id': STR_OR_INT, 'grid_region': STR, 'substation_name': STR,
        'feeder_line': STR, 'distribution_zone': STR, 'grid_operator': STR,
        'grid_voltage': NUM, 'grid_current': NUM, 'grid_load_kw': NUM,
        'transformer_load': NUM, 'line_loss_percent': NUM, 'load_variation': NUM,
        'frequency_variation': NUM, 'grid_capacity_kw': NUM,
        'demand_forecast_kw': NUM, 'reserve_margin': NUM,
        '_batch_date': STR, '_ingestion_ts': TS, '_source_file': STR, '_stream_name': STR,
    },
    bronze_tables['bronze_tariff_metrics']: {
        'household_id': STR_OR_INT, 'tariff_region': STR, 'tariff_city': STR,
        'tariff_plan_type': STR, 'billing_cycle': STR, 'utility_provider': STR,
        'unit_rate': NUM, 'peak_rate': NUM, 'offpeak_rate': NUM,
        'fixed_charge': NUM, 'tax_amount': NUM, 'subsidy_amount': NUM,
        'monthly_bill': NUM, 'billing_units': NUM,
        'late_fee': NUM, 'adjustment_amount': NUM,
        '_batch_date': STR, '_ingestion_ts': TS, '_source_file': STR, '_stream_name': STR,
    },
    bronze_tables['bronze_weather']: {
        'household_id': STR_OR_INT, 'weather_region': STR, 'weather_city': STR,
        'weather_station': STR, 'climate_zone': STR, 'condition_type': STR,
        'temperature_celsius': NUM, 'humidity_percent': NUM, 'wind_speed_kmh': NUM,
        'rainfall_mm': NUM, 'pressure_hpa': NUM, 'solar_radiation': NUM,
        'dew_point': NUM, 'uv_index': NUM, 'visibility_km': NUM,
        'cloud_cover_percent': NUM, 'timestamp': STR,
        '_batch_date': STR, '_ingestion_ts': TS, '_source_file': STR, '_stream_name': STR,
    },
    # ── Silver ───────────────────────────────────────────────────────────────
    silver_tables['silver_energy_usage']: {
    'household_id': STR_OR_INT, 'region_name': STR, 'city_name': STR,
    'meter_type': STR, 'customer_category': STR, 'grid_zone': STR,
    'voltage_reading': NUM, 'current_reading': NUM, 'active_power_kw': NUM,
    'reactive_power_kvar': NUM, 'energy_usage_kwh': NUM, 'frequency_hz': NUM,
    'load_factor': NUM, 'peak_demand_kw': NUM, 'offpeak_demand_kw': NUM,
    'daily_consumption_kwh': NUM,
    'event_timestamp': TS,
    },
    silver_tables['silver_device_metrics']: {
        'household_id': STR_OR_INT, 'device_category': STR, 'device_brand': STR,
        'device_model': STR, 'maintenance_status': STR, 'installation_region': STR,
        'runtime_hours': NUM, 'device_power_kw': NUM, 'motor_speed_rpm': NUM,
        'efficiency_ratio': NUM, 'energy_draw_kwh': NUM, 'heat_output': NUM,
        'cooling_load': NUM, 'device_voltage': NUM, 'device_current': NUM,
        'device_temperature': NUM,
    },
    silver_tables['silver_grid_load']: {
        'household_id': STR_OR_INT, 'grid_region': STR, 'substation_name': STR,
        'feeder_line': STR, 'distribution_zone': STR, 'grid_operator': STR,
        'grid_voltage': NUM, 'grid_current': NUM, 'grid_load_kw': NUM,
        'transformer_load': NUM, 'line_loss_percent': NUM, 'load_variation': NUM,
        'frequency_variation': NUM, 'grid_capacity_kw': NUM,
        'demand_forecast_kw': NUM, 'reserve_margin': NUM,
    },
    silver_tables['silver_tariff_metrics']: {
        'household_id': STR_OR_INT, 'tariff_region': STR, 'tariff_city': STR,
        'tariff_plan_type': STR, 'billing_cycle': STR, 'utility_provider': STR,
        'unit_rate': NUM, 'peak_rate': NUM, 'offpeak_rate': NUM,
        'fixed_charge': NUM, 'tax_amount': NUM, 'subsidy_amount': NUM,
        'monthly_bill': NUM, 'billing_units': NUM,
        'late_fee': NUM, 'adjustment_amount': NUM,
    },
    silver_tables['silver_weather']: {
    'household_id': STR_OR_INT, 'weather_region': STR, 'weather_city': STR,
    'weather_station': STR, 'climate_zone': STR, 'condition_type': STR,
    'temperature_celsius': NUM, 'humidity_percent': NUM, 'wind_speed_kmh': NUM,
    'rainfall_mm': NUM, 'pressure_hpa': NUM, 'solar_radiation': NUM,
    'dew_point': NUM, 'uv_index': NUM, 'visibility_km': NUM,
    'cloud_cover_percent': NUM,   # ← timestamp removed
    },
    # ── Gold dims ─────────────────────────────────────────────────────────────
    gold_tables['dim_household']: {
        'household_sk': INT, 'household_id': STR_OR_INT,
        'meter_type': STR, 'customer_category': STR,
        'billing_cycle': STR, 'utility_provider': STR,
        'city_name': STR, 'region_name': STR,
        'avg_monthly_bill': NUM, 'total_units_consumed': NUM,
    },
    gold_tables['dim_device']: {
        'device_sk': INT, 'device_category': STR, 'device_brand': STR,
        'device_model': STR, 'installation_region': STR, 'maintenance_status': STR,
        'avg_runtime_hours': NUM, 'avg_efficiency': NUM,
        'avg_power_consumption': NUM, 'max_device_temperature': NUM,
    },
    gold_tables['dim_location']: {
        'location_sk': INT, 'region_name': STR, 'city_name': STR,
        'climate_zone': STR, 'avg_temperature': NUM,
        'avg_humidity': NUM, 'avg_grid_load': NUM, 'peak_grid_load': NUM,
    },
    gold_tables['dim_tariff']: {
        'tariff_sk': INT, 'tariff_plan_type': STR, 'tariff_region': STR,
        'avg_unit_rate': NUM, 'avg_peak_rate': NUM,
        'avg_offpeak_rate': NUM, 'avg_fixed_charge': NUM, 'avg_subsidy': NUM,
    },
    gold_tables['fact_energy_consumption']: {
        'fact_sk': {'bigint', 'long'},
        'household_sk': INT, 'device_sk': INT, 'location_sk': INT, 'tariff_sk': INT,
        'energy_usage_kwh': NUM, 'active_power_kw': NUM,
        'reactive_power_kvar': NUM, 'peak_demand_kw': NUM, 'offpeak_demand_kw': NUM,
        'grid_load_kw': NUM, 'transformer_load': NUM, 'line_loss_percent': NUM,
        'temperature_celsius': NUM, 'humidity_percent': NUM,
        'monthly_bill': NUM, 'billing_units': NUM,
        'runtime_hours': NUM, 'energy_draw_kwh': NUM,
        'energy_cost': NUM, 'power_factor': NUM,
        'device_efficiency': NUM, 'grid_stress_ratio': NUM,
    },
}

def validate_schema(tbl_path, expected):
    df = load(tbl_path)
    actual = {f.name: f.dataType.simpleString().lower() for f in df.schema.fields}
    for cname, accepted in expected.items():
        assert cname in actual, \
            f'FAILED: {tbl_path} — column {cname!r} MISSING'
        assert any(t in actual[cname] for t in accepted), \
            f'FAILED: {tbl_path}.{cname} — type {actual[cname]!r} not in {accepted}'
    print(f'PASSED: {tbl_path} — schema valid ({len(expected)} columns checked)')

for tbl_path, expected in expected_schemas.items():
    validate_schema(tbl_path, expected)

print()
print('ALL SCHEMA VALIDATION CHECKS PASSED')

---
# **Part 4 — Count Validation**

From script execution logs: all 5 src tables = exactly 50,000 rows.

In [0]:
print('=' * 65)
print('COUNT VALIDATION — SRC → BRONZE → SILVER → GOLD')
print('=' * 65)

EXPECTED = 50_000

stream_triples = [
    ('src_energy_usage',   'bronze_energy_usage',   'silver_energy_usage'),
    ('src_device_metrics', 'bronze_device_metrics', 'silver_device_metrics'),
    ('src_grid_load',      'bronze_grid_load',      'silver_grid_load'),
    ('src_tariff_metrics', 'bronze_tariff_metrics', 'silver_tariff_metrics'),
    ('src_weather',        'bronze_weather',        'silver_weather'),
]

# Expected Silver ranges from script logs:
# energy_usage: ~48,995 (5 dedup + ~1000 null drop)
# device_metrics: ~49,824 (176 dedup)
# grid_load: variable dedup
# tariff_metrics: ~49,049 (951 dedup)
# weather: ~49,000 (0 dedup + ~1000 null drop)
silver_expected_ranges = {
    'silver_energy_usage'  : (48_000, 50_000),
    'silver_device_metrics': (49_500, 50_000),
    'silver_grid_load'     : (49_000, 50_000),
    'silver_tariff_metrics': (48_500, 50_000),
    'silver_weather'       : (48_000, 50_000),
}

sv_counts = {}

print('\n--- src tables: exactly 50,000 ---')
for s_key, b_key, sv_key in stream_triples:
    cnt = load(src_tables[s_key]).count()
    assert cnt == EXPECTED, f'FAILED: {s_key} has {cnt:,}, expected {EXPECTED:,}'
    print(f'PASSED: {s_key} — {cnt:,} rows')

print('\n--- bronze tables: exactly 50,000 (overwrite mode) ---')
for s_key, b_key, sv_key in stream_triples:
    cnt = load(bronze_tables[b_key]).count()
    assert cnt == EXPECTED, f'FAILED: {b_key} has {cnt:,}, expected {EXPECTED:,}'
    print(f'PASSED: {b_key} — {cnt:,} rows')

print('\n--- Silver: within expected range ---')
for s_key, b_key, sv_key in stream_triples:
    cnt = load(silver_tables[sv_key]).count()
    sv_counts[sv_key] = cnt
    lo, hi = silver_expected_ranges[sv_key]
    assert lo <= cnt <= hi, f'FAILED: {sv_key} has {cnt:,}, expected [{lo:,}, {hi:,}]'
    assert cnt <= EXPECTED, f'FAILED: {sv_key} ({cnt:,}) > Bronze ({EXPECTED:,})'
    retention = cnt / EXPECTED
    assert retention >= 0.95, f'FAILED: {sv_key} retention {retention:.1%} < 95%'
    print(f'PASSED: {sv_key} — {cnt:,} rows (retention: {retention:.2%})')

print('\n--- Gold fact = Silver energy_usage ---')
sv_eu = sv_counts['silver_energy_usage']
gf    = load(gold_tables['fact_energy_consumption']).count()
assert sv_eu == gf, f'FAILED: silver_energy_usage ({sv_eu:,}) != fact ({gf:,})'
print(f'PASSED: fact_energy_consumption ({gf:,}) = silver_energy_usage ({sv_eu:,})')

print('\n--- Gold dimension table sizes ---')
for dim_key in ['dim_household', 'dim_device', 'dim_location', 'dim_tariff']:
    cnt = load(gold_tables[dim_key]).count()
    assert 1 <= cnt <= sv_eu
    print(f'PASSED: {dim_key} — {cnt:,} rows')

print('\n--- Metrics grain checks ---')
hh_cnt = load(gold_tables['dim_household']).count()
tiers  = load(metrics_tables['metrics_household_consumption_tiers']).count()
assert tiers == hh_cnt, f'FAILED: household_tiers {tiers:,} != dim_household {hh_cnt:,}'
print(f'PASSED: metrics_household_consumption_tiers — {tiers:,} rows (= dim_household)')

plan_cnt = load(gold_tables['dim_tariff']).count()
tariff_m = load(metrics_tables['metrics_tariff_plan_comparison']).count()
assert tariff_m == plan_cnt, f'FAILED: tariff_comparison {tariff_m} != dim_tariff {plan_cnt}'
print(f'PASSED: metrics_tariff_plan_comparison — {tariff_m} rows (= dim_tariff)')

for m_key in ['metrics_regional_energy_summary', 'metrics_device_efficiency_ranking',
              'metrics_grid_health_overview', 'metrics_weather_energy_correlation']:
    cnt = load(metrics_tables[m_key]).count()
    assert cnt > 0
    print(f'PASSED: {m_key} — {cnt} rows')

print()
print('ALL COUNT VALIDATION CHECKS PASSED')

---
# **Part 5 — Write Pytest File**

In [0]:
import pytest

test_code = '''
import pytest
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, round as spark_round

CAT="energy_catalog"; BR="raw"; SV="processed"; GO="analytics"
EXPECTED=50_000; TOL=0.95
NUM={"double","float"}
INT={"int","integer","bigint","long"}
STR_OR_INT={"string","int","integer","bigint","long"}

@pytest.fixture(scope="session")
def spark(): return SparkSession.builder.appName("energy-pipeline-pytest").getOrCreate()

def t(spark,s,n): return spark.table(f"{CAT}.{s}.{n}")

def chk(spark,s,n,e):
    a={f.name:f.dataType.simpleString().lower() for f in t(spark,s,n).schema.fields}
    for cn,ac in e.items():
        assert cn in a, f"{s}.{n}: column {cn!r} MISSING"
        assert any(x in a[cn] for x in ac), f"{s}.{n}.{cn}: {a[cn]!r} not in {ac}"

# ══════════════════════════════════════════════════
# DATA QUALITY — TABLE EXISTENCE
# ══════════════════════════════════════════════════

@pytest.mark.parametrize("n",["bronze_energy_usage","bronze_device_metrics",
    "bronze_grid_load","bronze_tariff_metrics","bronze_weather"])
def test_bronze_not_empty(spark,n): assert t(spark,BR,n).count()>0

@pytest.mark.parametrize("n",["silver_energy_usage","silver_device_metrics",
    "silver_grid_load","silver_tariff_metrics","silver_weather"])
def test_silver_not_empty(spark,n): assert t(spark,SV,n).count()>0

@pytest.mark.parametrize("n",["dim_household","dim_device","dim_location","dim_tariff",
    "fact_energy_consumption","metrics_regional_energy_summary",
    "metrics_household_consumption_tiers","metrics_device_efficiency_ranking",
    "metrics_tariff_plan_comparison","metrics_grid_health_overview",
    "metrics_weather_energy_correlation"])
def test_gold_not_empty(spark,n): assert t(spark,GO,n).count()>0

# ══════════════════════════════════════════════════
# DATA QUALITY — BRONZE AUDIT COLUMNS
# ══════════════════════════════════════════════════

@pytest.mark.parametrize("n",["bronze_energy_usage","bronze_device_metrics",
    "bronze_grid_load","bronze_tariff_metrics","bronze_weather"])
def test_bronze_audit_cols_present(spark,n):
    df=t(spark,BR,n)
    for ac in ["_batch_date","_ingestion_ts","_source_file","_stream_name"]:
        assert ac in df.columns, f"{n} missing audit column {ac}"

@pytest.mark.parametrize("n",["silver_energy_usage","silver_device_metrics",
    "silver_grid_load","silver_tariff_metrics","silver_weather"])
def test_silver_audit_cols_dropped(spark,n):
    df=t(spark,SV,n)
    for ac in ["_batch_date","_ingestion_ts","_source_file","_stream_name"]:
        assert ac not in df.columns, f"{n} still contains audit column {ac}"

# ══════════════════════════════════════════════════
# DATA QUALITY — DEDUPLICATION
# Mirrors exact dropDuplicates() logic from silver_layer.ipynb
# ══════════════════════════════════════════════════

def test_silver_energy_usage_no_full_row_duplicates(spark):
    """silver_layer.ipynb: df.dropDuplicates()
    Full-row dedup, no key. Multiple readings per household per timestamp are valid."""
    df=t(spark,SV,"silver_energy_usage")
    total=df.count()
    distinct=df.distinct().count()
    assert total==distinct, \
        f"silver_energy_usage has {total-distinct} full-row duplicate rows after dedup"

def test_silver_device_metrics_no_duplicates(spark):
    """silver_layer.ipynb:
    df.withColumn("runtime_hours_rounded", spark_round("runtime_hours", 0))
      .dropDuplicates(["household_id","device_category","device_brand",
                       "device_model","runtime_hours_rounded"])
      .drop("runtime_hours_rounded")"""
    df=t(spark,SV,"silver_device_metrics") \
        .withColumn("runtime_hours_rounded",spark_round("runtime_hours",0))
    dup=(df.groupBy("household_id","device_category","device_brand",
                    "device_model","runtime_hours_rounded")
          .count().filter(col("count")>1).count())
    assert dup==0, f"silver_device_metrics has {dup} duplicates on script dedup key"

def test_silver_grid_load_no_duplicates(spark):
    """silver_layer.ipynb:
    df.dropDuplicates(["household_id","grid_region","substation_name",
                       "feeder_line","distribution_zone","grid_operator"])"""
    dup=(t(spark,SV,"silver_grid_load")
         .groupBy("household_id","grid_region","substation_name",
                  "feeder_line","distribution_zone","grid_operator")
         .count().filter(col("count")>1).count())
    assert dup==0, f"silver_grid_load has {dup} duplicates on infrastructure identity key"

def test_silver_tariff_metrics_no_duplicates(spark):
    """silver_layer.ipynb:
    df.dropDuplicates(["household_id","tariff_region","tariff_city",
                       "tariff_plan_type","utility_provider"])"""
    dup=(t(spark,SV,"silver_tariff_metrics")
         .groupBy("household_id","tariff_region","tariff_city",
                  "tariff_plan_type","utility_provider")
         .count().filter(col("count")>1).count())
    assert dup==0, f"silver_tariff_metrics has {dup} duplicates on business key"

def test_silver_weather_no_full_row_duplicates(spark):
    """silver_layer.ipynb: df.dropDuplicates()
    Full-row dedup, no key. 0 duplicate rows expected in weather stream."""
    df=t(spark,SV,"silver_weather")
    total=df.count()
    distinct=df.distinct().count()
    assert total==distinct, \
        f"silver_weather has {total-distinct} full-row duplicate rows after dedup"

# ══════════════════════════════════════════════════
# DATA QUALITY — NULLS
# ══════════════════════════════════════════════════

@pytest.mark.parametrize("n,cols",[
    ("silver_energy_usage",["household_id","region_name","energy_usage_kwh",
                            "active_power_kw","event_timestamp"]),
    ("silver_device_metrics",["household_id","device_category",
                              "device_brand","device_model"]),
    ("silver_grid_load",["household_id","grid_region",
                         "grid_load_kw","grid_capacity_kw"]),
    ("silver_tariff_metrics",["household_id","tariff_plan_type",
                              "billing_cycle","monthly_bill"]),
    ("silver_weather",["household_id","weather_region",
                       "temperature_celsius","condition_type"]),
])
def test_silver_no_nulls(spark,n,cols):
    df=t(spark,SV,n)
    for c in cols:
        assert df.filter(col(c).isNull()).count()==0, f"{n}.{c} has nulls"

# ══════════════════════════════════════════════════
# DATA QUALITY — REFERENTIAL INTEGRITY
# ══════════════════════════════════════════════════

def test_fact_sk_integrity_household(spark):
    assert t(spark,GO,"fact_energy_consumption").join(
        t(spark,GO,"dim_household"),"household_sk","left_anti").count()==0

def test_fact_sk_integrity_location(spark):
    assert t(spark,GO,"fact_energy_consumption").join(
        t(spark,GO,"dim_location"),"location_sk","left_anti").count()==0

def test_fact_sk_integrity_device(spark):
    assert t(spark,GO,"fact_energy_consumption").join(
        t(spark,GO,"dim_device"),"device_sk","left_anti").count()==0

def test_fact_sk_integrity_tariff(spark):
    assert t(spark,GO,"fact_energy_consumption").join(
        t(spark,GO,"dim_tariff"),"tariff_sk","left_anti").count()==0

# ══════════════════════════════════════════════════
# DATA QUALITY — RECONCILIATION
# ══════════════════════════════════════════════════

def test_reconciliation_row_count(spark):
    s=t(spark,SV,"silver_energy_usage").count()
    g=t(spark,GO,"fact_energy_consumption").count()
    assert s==g, f"silver={s:,} fact={g:,}"

def test_reconciliation_energy_sum(spark):
    s=t(spark,SV,"silver_energy_usage") \
        .agg(spark_sum("energy_usage_kwh").alias("x")).collect()[0]["x"]
    g=t(spark,GO,"fact_energy_consumption") \
        .agg(spark_sum("energy_usage_kwh").alias("x")).collect()[0]["x"]
    assert round(s,2)==round(g,2), f"energy sum mismatch silver={s} gold={g}"

# ══════════════════════════════════════════════════
# UNIT TESTS — KEY TRANSFORMATION LOGIC
# ══════════════════════════════════════════════════

def test_event_timestamp_exists_no_nulls(spark):
    """silver_layer.ipynb: to_timestamp(col("timestamp"), "dd-MM-yyyy HH:mm") -> event_timestamp"""
    df=t(spark,SV,"silver_energy_usage")
    assert "event_timestamp" in df.columns, "event_timestamp column missing"
    assert df.filter(col("event_timestamp").isNull()).count()==0, \
        "event_timestamp has nulls — timestamp parsing failed"

def test_efficiency_ratio_clamped_05_10(spark):
    """silver_layer.ipynb: when(col < 0.5, 0.5).when(col > 1.0, 1.0).otherwise(col)"""
    df=t(spark,SV,"silver_device_metrics")
    assert df.filter(col("efficiency_ratio")<0.5).count()==0, \
        "efficiency_ratio below 0.5 after clamping"
    assert df.filter(col("efficiency_ratio")>1.0).count()==0, \
        "efficiency_ratio above 1.0 after clamping"

def test_load_variation_negatives_preserved(spark):
    """silver_layer.ipynb: legitimate grid measurement — must NOT be filtered out"""
    assert t(spark,SV,"silver_grid_load").filter(col("load_variation")<0).count()>0, \
        "No negative load_variation found — expected signed values (-20 to +20)"

def test_frequency_variation_negatives_preserved(spark):
    """silver_layer.ipynb: legitimate grid measurement — must NOT be filtered out"""
    assert t(spark,SV,"silver_grid_load").filter(col("frequency_variation")<0).count()>0, \
        "No negative frequency_variation found — expected signed values (-0.5 to +0.5)"

def test_adjustment_amount_negatives_preserved(spark):
    """silver_layer.ipynb: billing credits — must NOT be filtered out"""
    assert t(spark,SV,"silver_tariff_metrics").filter(col("adjustment_amount")<0).count()>0, \
        "No negative adjustment_amount found — billing credits should exist"

def test_condition_type_no_pct(spark):
    """silver_layer.ipynb: regexp_replace(col("condition_type"), "%", "")"""
    assert t(spark,SV,"silver_weather").filter(
        col("condition_type").contains("%")).count()==0, \
        "% suffix still present in condition_type"

def test_condition_type_valid_values(spark):
    """silver_layer.ipynb: trim(upper(...)) + % removed -> CLOUDY, RAINY, SUNNY only"""
    invalid=t(spark,SV,"silver_weather").filter(
        ~col("condition_type").isin("CLOUDY","RAINY","SUNNY")).count()
    assert invalid==0, f"{invalid} rows have unexpected condition_type value"

def test_energy_usage_null_drop(spark):
    """silver_layer.ipynb: dropna(subset=[10 numeric cols]) removes ~1000 rows"""
    df=t(spark,SV,"silver_energy_usage")
    for c in ["voltage_reading","current_reading","active_power_kw",
              "reactive_power_kvar","energy_usage_kwh","frequency_hz",
              "load_factor","peak_demand_kw","offpeak_demand_kw",
              "daily_consumption_kwh"]:
        assert df.filter(col(c).isNull()).count()==0, \
            f"silver_energy_usage.{c} still has nulls after dropna"

@pytest.mark.parametrize("dc",["energy_cost","power_factor",
                                "device_efficiency","grid_stress_ratio"])
def test_fact_derived_col_exists(spark,dc):
    """gold_layer.ipynb: withColumn() derived columns"""
    assert dc in t(spark,GO,"fact_energy_consumption").columns, \
        f"fact_energy_consumption missing derived column {dc}"

def test_energy_cost_no_nulls_no_neg(spark):
    """gold_layer.ipynb: round(energy_usage_kwh * avg_unit_rate, 4)"""
    df=t(spark,GO,"fact_energy_consumption")
    assert df.filter(col("energy_cost").isNull()).count()==0, \
        "energy_cost has null values"
    assert df.filter(col("energy_cost")<0).count()==0, \
        "energy_cost has negative values"

@pytest.mark.parametrize("sk",["fact_sk","household_sk","device_sk",
                                "location_sk","tariff_sk"])
def test_fact_sk_not_null(spark,sk):
    """gold_layer.ipynb: all surrogate keys must be populated"""
    assert t(spark,GO,"fact_energy_consumption").filter(
        col(sk).isNull()).count()==0, f"fact.{sk} has null values"

def test_consumption_tier_valid_values(spark):
    """gold_layer.ipynb: percent_rank() tiers -> Low / Medium / High / Very High"""
    invalid=t(spark,GO,"metrics_household_consumption_tiers").filter(
        ~col("consumption_tier").isin("Low","Medium","High","Very High")).count()
    assert invalid==0, f"{invalid} rows have invalid consumption_tier"

def test_efficiency_grade_valid_values(spark):
    """gold_layer.ipynb: efficiency_ratio thresholds -> A / B / C / D"""
    invalid=t(spark,GO,"metrics_device_efficiency_ranking").filter(
        ~col("efficiency_grade").isin("A","B","C","D")).count()
    assert invalid==0, f"{invalid} rows have invalid efficiency_grade"

# ══════════════════════════════════════════════════
# SCHEMA VALIDATION
# ══════════════════════════════════════════════════

@pytest.mark.parametrize("n,e",[
    ("bronze_energy_usage",{
        "household_id":STR_OR_INT,"region_name":{"string"},"city_name":{"string"},
        "energy_usage_kwh":NUM,"active_power_kw":NUM,"frequency_hz":NUM,
        "load_factor":NUM,"timestamp":{"string"},
        "_batch_date":{"string"},"_ingestion_ts":{"timestamp","string"},
        "_source_file":{"string"},"_stream_name":{"string"},
    }),
    ("bronze_device_metrics",{
        "household_id":STR_OR_INT,"device_category":{"string"},
        "efficiency_ratio":NUM,"energy_draw_kwh":NUM,"device_temperature":NUM,
        "_batch_date":{"string"},"_ingestion_ts":{"timestamp","string"},
        "_source_file":{"string"},"_stream_name":{"string"},
    }),
    ("bronze_grid_load",{
        "household_id":STR_OR_INT,"grid_region":{"string"},
        "grid_load_kw":NUM,"load_variation":NUM,"frequency_variation":NUM,
        "_batch_date":{"string"},"_ingestion_ts":{"timestamp","string"},
        "_source_file":{"string"},"_stream_name":{"string"},
    }),
    ("bronze_tariff_metrics",{
        "household_id":STR_OR_INT,"tariff_plan_type":{"string"},
        "monthly_bill":NUM,"adjustment_amount":NUM,
        "_batch_date":{"string"},"_ingestion_ts":{"timestamp","string"},
        "_source_file":{"string"},"_stream_name":{"string"},
    }),
    ("bronze_weather",{
        "household_id":STR_OR_INT,"condition_type":{"string"},
        "temperature_celsius":NUM,"humidity_percent":NUM,"timestamp":{"string"},
        "_batch_date":{"string"},"_ingestion_ts":{"timestamp","string"},
        "_source_file":{"string"},"_stream_name":{"string"},
    }),
])
def test_bronze_schema(spark,n,e): chk(spark,BR,n,e)

@pytest.mark.parametrize("n,e",[
    ("silver_energy_usage",{
        "household_id":STR_OR_INT,"region_name":{"string"},
        "energy_usage_kwh":NUM,"active_power_kw":NUM,
        "load_factor":NUM,"frequency_hz":NUM,
        "event_timestamp":{"timestamp","string"},
    }),
    ("silver_device_metrics",{
        "household_id":STR_OR_INT,"device_category":{"string"},
        "efficiency_ratio":NUM,"energy_draw_kwh":NUM,"device_temperature":NUM,
    }),
    ("silver_grid_load",{
        "household_id":STR_OR_INT,"grid_region":{"string"},
        "grid_load_kw":NUM,"grid_capacity_kw":NUM,
        "load_variation":NUM,"frequency_variation":NUM,"reserve_margin":NUM,
    }),
    ("silver_tariff_metrics",{
        "household_id":STR_OR_INT,"tariff_plan_type":{"string"},
        "monthly_bill":NUM,"billing_units":NUM,
        "adjustment_amount":NUM,"unit_rate":NUM,
    }),
    ("silver_weather",{
        "household_id":STR_OR_INT,"condition_type":{"string"},
        "temperature_celsius":NUM,"humidity_percent":NUM,"climate_zone":{"string"},
    }),
])
def test_silver_schema(spark,n,e): chk(spark,SV,n,e)

@pytest.mark.parametrize("n,e",[
    ("dim_household",{
        "household_sk":INT,"household_id":STR_OR_INT,
        "region_name":{"string"},"city_name":{"string"},"avg_monthly_bill":NUM,
    }),
    ("dim_device",{
        "device_sk":INT,"device_category":{"string"},
        "avg_efficiency":NUM,"avg_power_consumption":NUM,
    }),
    ("dim_location",{
        "location_sk":INT,"region_name":{"string"},
        "climate_zone":{"string"},"avg_temperature":NUM,"avg_grid_load":NUM,
    }),
    ("dim_tariff",{
        "tariff_sk":INT,"tariff_plan_type":{"string"},"avg_unit_rate":NUM,
    }),
    ("fact_energy_consumption",{
        "fact_sk":{"bigint","long"},
        "household_sk":INT,"device_sk":INT,"location_sk":INT,"tariff_sk":INT,
        "energy_usage_kwh":NUM,"active_power_kw":NUM,
        "peak_demand_kw":NUM,"offpeak_demand_kw":NUM,
        "energy_cost":NUM,"power_factor":NUM,
        "device_efficiency":NUM,"grid_stress_ratio":NUM,"monthly_bill":NUM,
    }),
])
def test_gold_schema(spark,n,e): chk(spark,GO,n,e)

# ══════════════════════════════════════════════════
# COUNT VALIDATION
# ══════════════════════════════════════════════════

@pytest.mark.parametrize("n",["src_energy_usage","src_device_metrics","src_grid_load",
    "src_tariff_metrics","src_weather"])
def test_src_exactly_50k(spark,n):
    assert t(spark,BR,n).count()==EXPECTED

@pytest.mark.parametrize("n",["bronze_energy_usage","bronze_device_metrics",
    "bronze_grid_load","bronze_tariff_metrics","bronze_weather"])
def test_bronze_exactly_50k(spark,n):
    assert t(spark,BR,n).count()==EXPECTED

@pytest.mark.parametrize("n,lo,hi",[
    ("silver_energy_usage",  48_000,50_000),
    ("silver_device_metrics",49_500,50_000),
    ("silver_grid_load",     49_000,50_000),
    ("silver_tariff_metrics",48_500,50_000),
    ("silver_weather",       48_000,50_000),
])
def test_silver_range(spark,n,lo,hi):
    cnt=t(spark,SV,n).count()
    assert lo<=cnt<=hi, f"{n}: {cnt:,} outside [{lo:,},{hi:,}]"
    assert cnt/EXPECTED>=TOL, f"{n}: retention {cnt/EXPECTED:.1%} below 95%"

def test_fact_equals_silver_energy_usage(spark):
    s=t(spark,SV,"silver_energy_usage").count()
    g=t(spark,GO,"fact_energy_consumption").count()
    assert s==g, f"silver_energy_usage={s:,}, fact_energy_consumption={g:,}"

def test_household_tiers_grain(spark):
    assert t(spark,GO,"metrics_household_consumption_tiers").count() \
        ==t(spark,GO,"dim_household").count()

def test_tariff_comparison_grain(spark):
    assert t(spark,GO,"metrics_tariff_plan_comparison").count() \
        ==t(spark,GO,"dim_tariff").count()
'''

## **Cell 7 — Run Pytest**

In [0]:
result = pytest.main(["-v", "--tb=short", "--cache-clear",
                      "/tmp/test_energy_pipeline_full.py"])
print("Pytest exit code:", result)
assert result == 0, "One or more tests FAILED"